<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/Copy_of_structure_esm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import requests
import time
import warnings
!pip install biopython
import Bio
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
warnings.filterwarnings('ignore')


In [ ]:
pairs_mapped = pd.read_parquet('/content/pairs_mapped.parquet')
display(pairs_mapped.head())

,protein1,protein2,label,source
0,P26437,P18085,1,STRING_experimental
1,P26437,P10947,1,STRING_experimental
2,P20645,Q15017,1,STRING_experimental
3,Q02790,Q5VVC3,1,STRING_experimental
4,Q02790,P42345,1,STRING_experimental


In [ ]:
unique_proteins = set(pairs_mapped['protein1']) | set(pairs_mapped['protein2'])
print(f"Unique proteins needing structures: {len(unique_proteins):,}")

Unique proteins needing structures: 4,403


In [ ]:
import os

# Ensure directory exists
os.makedirs('data/raw', exist_ok=True)

# Verify it was created
print(os.path.exists('data/raw'))
print(os.listdir('data') if os.path.exists('data') else "data/ does not exist")

True
['raw']


In [ ]:
!wget -O data/raw/pdb_chain_uniprot.tsv.gz \
  "https://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/tsv/pdb_chain_uniprot.tsv.gz"

--2026-06-18 05:41:27--  https://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/tsv/pdb_chain_uniprot.tsv.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5981310 (5.7M) [application/x-gzip]
Saving to: ‘data/raw/pdb_chain_uniprot.tsv.gz’

data/raw/pdb_chain_ 100%[===================>]   5.70M  1.23MB/s    in 4.7s    

2026-06-18 05:41:33 (1.21 MB/s) - ‘data/raw/pdb_chain_uniprot.tsv.gz’ saved [5981310/5981310]



In [ ]:
import pandas as pd

sifts = pd.read_csv(
    'data/raw/pdb_chain_uniprot.tsv.gz',
    sep='\t',
    compression='gzip',
    skiprows=1,
    low_memory=False
)

print(f"SIFTS shape: {sifts.shape}")

SIFTS shape: (993923, 9)


In [ ]:
# Filter SIFTS to only your corrected protein set
sifts_filtered = sifts[sifts['SP_PRIMARY'].isin(unique_proteins)].copy()
print(f"SIFTS rows matching your proteins: {len(sifts_filtered):,}")

# Calculate coverage and pick best PDB chain per protein
sifts_filtered['coverage'] = sifts_filtered['SP_END'] - sifts_filtered['SP_BEG']

best_pdb = sifts_filtered.loc[
    sifts_filtered.groupby('SP_PRIMARY')['coverage'].idxmax()
].copy()

best_pdb = best_pdb[['SP_PRIMARY', 'PDB', 'CHAIN', 'coverage']].reset_index(drop=True)
best_pdb.columns = ['uniprot_id', 'pdb_id', 'chain', 'coverage']

print(f"\nBest PDB selected for: {len(best_pdb):,} proteins")
print(f"Proteins with NO PDB structure: {len(unique_proteins) - len(best_pdb):,}")

SIFTS rows matching your proteins: 105,928

Best PDB selected for: 2,133 proteins
Proteins with NO PDB structure: 2,270


In [ ]:
# Build the complete structure manifest
all_proteins = pd.DataFrame({'uniprot_id': list(unique_proteins)})

structure_manifest = all_proteins.merge(
    best_pdb[['uniprot_id', 'pdb_id', 'chain']],
    on='uniprot_id',
    how='left'
)

structure_manifest['structure_source'] = structure_manifest['pdb_id'].apply(
    lambda x: 'PDB' if pd.notna(x) else 'AF2'
)

print(f"Total proteins:         {len(structure_manifest):,}")
print(f"Using PDB:              {(structure_manifest['structure_source']=='PDB').sum():,}")
print(f"Using AF2:              {(structure_manifest['structure_source']=='AF2').sum():,}")
print(f"\nFirst 5 rows:")
print(structure_manifest.head())

Total proteins:         4,403
Using PDB:              2,133
Using AF2:              2,270

First 5 rows:
  uniprot_id pdb_id chain structure_source
0     Q99576    NaN   NaN              AF2
1     Q9Y3D5   3j9m    AP              PDB
2     P00813   7rtg     A              PDB
3     P63146   1jas     A              PDB
4     Q13287   21nh     A              PDB


In [ ]:
import json
from datetime import datetime

structure_manifest.to_csv('data/raw/structure_manifest.tsv', sep='\t', index=False)
structure_manifest.to_parquet('data/raw/structure_manifest.parquet', index=False)

metadata = {
    "date_created":     datetime.today().strftime('%Y-%m-%d'),
    "total_proteins":   len(structure_manifest),
    "pdb_count":        int((structure_manifest['structure_source']=='PDB').sum()),
    "af2_count":        int((structure_manifest['structure_source']=='AF2').sum()),
    "selection_criterion": "Maximum UniProt sequence coverage per protein",
    "correction_note": "Rebuilt after correcting Negatome organism contamination and fixing the ID-mapping bug. Protein universe reduced from 6,539 to 4,403."
}

with open('data/raw/structure_manifest_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully")
print(json.dumps(metadata, indent=2))

Saved successfully
{
  "date_created": "2026-06-18",
  "total_proteins": 4403,
  "pdb_count": 2133,
  "af2_count": 2270,
  "selection_criterion": "Maximum UniProt sequence coverage per protein",
  "correction_note": "Rebuilt after correcting Negatome organism contamination and fixing the ID-mapping bug. Protein universe reduced from 6,539 to 4,403."
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

base_dir = '/content/drive/MyDrive/ppi_gnn'
os.makedirs(f'{base_dir}/data/raw', exist_ok=True)

# All corrected files from this notebook's /content/ and /content/data/raw/
files_to_copy = {
    '/content/pairs_combined.tsv':                  f'{base_dir}/data/raw/pairs_combined.tsv',
    '/content/pairs_combined.parquet':               f'{base_dir}/data/raw/pairs_combined.parquet',
    '/content/pairs_combined_metadata.json':         f'{base_dir}/data/raw/pairs_combined_metadata.json',
    '/content/negatome_human.tsv':                   f'{base_dir}/data/raw/negatome_human.tsv',
    '/content/negatome_human.parquet':                f'{base_dir}/data/raw/negatome_human.parquet',
    '/content/negatome_human_metadata.json':          f'{base_dir}/data/raw/negatome_human_metadata.json',
    '/content/pairs_mapped.tsv':                      f'{base_dir}/data/raw/pairs_mapped.tsv',
    '/content/pairs_mapped.parquet':                  f'{base_dir}/data/raw/pairs_mapped.parquet',
    '/content/pairs_mapped_metadata.json':             f'{base_dir}/data/raw/pairs_mapped_metadata.json',
    'data/raw/structure_manifest.tsv':                f'{base_dir}/data/raw/structure_manifest.tsv',
    'data/raw/structure_manifest.parquet':            f'{base_dir}/data/raw/structure_manifest.parquet',
    'data/raw/structure_manifest_metadata.json':      f'{base_dir}/data/raw/structure_manifest_metadata.json',
}

for src, dst in files_to_copy.items():
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied: {os.path.basename(src)}")
    else:
        print(f"NOT FOUND: {src}")

Copied: pairs_combined.tsv
Copied: pairs_combined.parquet
Copied: pairs_combined_metadata.json
Copied: negatome_human.tsv
Copied: negatome_human.parquet
Copied: negatome_human_metadata.json
Copied: pairs_mapped.tsv
Copied: pairs_mapped.parquet
Copied: pairs_mapped_metadata.json
Copied: structure_manifest.tsv
Copied: structure_manifest.parquet
Copied: structure_manifest_metadata.json


In [ ]:


def fetch_pdb_structure(pdb_id, save_path):
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return True
        return False
    except Exception:
        return False


def fetch_af2_structure(uniprot_id, save_path):
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    try:
        api_response = requests.get(api_url, timeout=15)
        if api_response.status_code != 200:
            return False
        data = api_response.json()
        if not data:
            return False
        pdb_url = data[0]['pdbUrl']
        pdb_response = requests.get(pdb_url, timeout=15)
        if pdb_response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(pdb_response.content)
            return True
        return False
    except Exception:
        return False


def parse_pdb_chain(pdb_path, chain_id):
    """For experimental PDB structures — extracts one specific chain."""
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure('protein', pdb_path)
    except Exception:
        return None

    model = structure[0]
    if chain_id not in model:
        return None

    chain = model[chain_id]
    residue_names, ca_coords, bfactors = [], [], []

    for residue in chain:
        if not is_aa(residue, standard=True):
            continue
        if 'CA' not in residue:
            continue
        residue_names.append(residue.get_resname())
        ca_coords.append(residue['CA'].get_coord().tolist())
        bfactors.append(residue['CA'].get_bfactor())   # resolution proxy for PDB, unused but stored

    if len(ca_coords) == 0:
        return None

    return {
        'residue_names': residue_names,
        'ca_coords': ca_coords,
        'n_residues': len(ca_coords),
        'plddt': None   # not applicable for PDB
    }


def parse_af2_structure(pdb_path):
    """For AF2 structures — single chain, B-factor column IS the pLDDT score."""
    parser = PDBParser(QUIET=True)
    try:
        structure = parser.get_structure('protein', pdb_path)
    except Exception:
        return None

    model = structure[0]
    chains = list(model)
    if len(chains) == 0:
        return None

    chain = chains[0]   # AF2 files have exactly one chain
    residue_names, ca_coords, plddt = [], [], []

    for residue in chain:
        if not is_aa(residue, standard=True):
            continue
        if 'CA' not in residue:
            continue
        residue_names.append(residue.get_resname())
        ca_coords.append(residue['CA'].get_coord().tolist())
        plddt.append(residue['CA'].get_bfactor())   # AF2 stores pLDDT here

    if len(ca_coords) == 0:
        return None

    return {
        'residue_names': residue_names,
        'ca_coords': ca_coords,
        'n_residues': len(ca_coords),
        'plddt': plddt
    }

print("Functions defined")

Functions defined


In [ ]:
# Test PDB path
test_pdb_row = structure_manifest[structure_manifest['structure_source']=='PDB'].iloc[0]
success = fetch_pdb_structure(test_pdb_row['pdb_id'], '/content/test.pdb')
if success:
    result = parse_pdb_chain('/content/test.pdb', test_pdb_row['chain'])
    print(f"PDB test ({test_pdb_row['uniprot_id']}): {result['n_residues']} residues" if result else "PDB parse failed")

# Test AF2 path
test_af2_row = structure_manifest[structure_manifest['structure_source']=='AF2'].iloc[0]
success = fetch_af2_structure(test_af2_row['uniprot_id'], '/content/test_af2.pdb')
if success:
    result = parse_af2_structure('/content/test_af2.pdb')
    print(f"AF2 test ({test_af2_row['uniprot_id']}): {result['n_residues']} residues, mean pLDDT={sum(result['plddt'])/len(result['plddt']):.1f}" if result else "AF2 parse failed")
else:
    print(f"AF2 fetch failed for {test_af2_row['uniprot_id']}")

AF2 test (Q99576): 134 residues, mean pLDDT=71.5


In [ ]:
import os
import json
import time

base_dir = '/content/drive/MyDrive/ppi_gnn'
checkpoint_path = f'{base_dir}/data/raw/structure_data_checkpoint.json'
progress_path   = f'{base_dir}/data/raw/structure_progress.json'

# Load existing progress if resuming
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'r') as f:
        structure_data = json.load(f)
    print(f"Resuming — {len(structure_data)} proteins already processed")
else:
    structure_data = {}
    print("Starting fresh")

proteins_to_process = [
    uid for uid in structure_manifest['uniprot_id']
    if uid not in structure_data
]
print(f"Proteins remaining to process: {len(proteins_to_process):,}")

Resuming — 4403 proteins already processed
Proteins remaining to process: 0


In [ ]:
PLDDT_THRESHOLD = 70.0
manifest_lookup = structure_manifest.set_index('uniprot_id').to_dict('index')

processed_count = 0
checkpoint_every = 100

for uid in proteins_to_process:
    row = manifest_lookup[uid]
    source = row['structure_source']

    status = 'failed'
    n_residues = None
    quality_note = None

    if source == 'PDB':
        tmp_path = '/content/tmp_structure.pdb'
        if fetch_pdb_structure(row['pdb_id'], tmp_path):
            result = parse_pdb_chain(tmp_path, row['chain'])
            if result and result['n_residues'] >= 20:   # minimum length sanity check
                status = 'success'
                n_residues = result['n_residues']
            else:
                quality_note = 'too_short_or_parse_failed'
        else:
            quality_note = 'download_failed'
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

    elif source == 'AF2':
        tmp_path = '/content/tmp_structure.pdb'
        if fetch_af2_structure(uid, tmp_path):
            result = parse_af2_structure(tmp_path)
            if result and result['n_residues'] >= 20:
                mean_plddt = sum(result['plddt']) / len(result['plddt'])
                if mean_plddt >= PLDDT_THRESHOLD:
                    status = 'success'
                    n_residues = result['n_residues']
                else:
                    quality_note = f'low_plddt_{mean_plddt:.1f}'
            else:
                quality_note = 'too_short_or_parse_failed'
        else:
            quality_note = 'download_failed'
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

    structure_data[uid] = {
        'source': source,
        'status': status,
        'n_residues': n_residues,
        'quality_note': quality_note
    }

    processed_count += 1

    if processed_count % checkpoint_every == 0:
        with open(checkpoint_path, 'w') as f:
            json.dump(structure_data, f)
        n_success = sum(1 for v in structure_data.values() if v['status']=='success')
        print(f"  Processed {processed_count}/{len(proteins_to_process)} this run | Total success so far: {n_success}/{len(structure_data)}")

    time.sleep(0.1)   # polite pacing for both RCSB and AlphaFold APIs

# Final save
with open(checkpoint_path, 'w') as f:
    json.dump(structure_data, f)

print(f"\nDone. Total processed: {len(structure_data)}")
n_success = sum(1 for v in structure_data.values() if v['status']=='success')
print(f"Successful: {n_success}")
print(f"Failed: {len(structure_data) - n_success}")


Done. Total processed: 4403
Successful: 2242
Failed: 2161


In [ ]:
import random

random.seed(123)

af2_failed_proteins = [
    uid for uid, v in structure_data.items()
    if v['source'] == 'AF2' and v['status'] == 'failed' and v['quality_note'] == 'download_failed'
]

print(f"Total AF2 download_failed proteins: {len(af2_failed_proteins):,}")

retry_sample = random.sample(af2_failed_proteins, 50)
print(f"Retesting a random sample of {len(retry_sample)}...\n")

Total AF2 download_failed proteins: 1,656
Retesting a random sample of 50...



In [ ]:
import time

retry_results = {}

for uid in retry_sample:
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uid}"
    try:
        resp = requests.get(api_url, timeout=10)
        retry_results[uid] = resp.status_code
    except Exception as e:
        retry_results[uid] = f"ERROR: {e}"
    time.sleep(0.2)

from collections import Counter
status_counts = Counter(retry_results.values())

print("Retry results breakdown:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

Retry results breakdown:
  404: 50


In [ ]:
successful_proteins = {
    uid for uid, v in structure_data.items() if v['status'] == 'success'
}

print(f"Proteins with usable structures: {len(successful_proteins):,}")

Proteins with usable structures: 2,242


In [ ]:
pairs_final = pairs_mapped[
    pairs_mapped['protein1'].isin(successful_proteins) &
    pairs_mapped['protein2'].isin(successful_proteins)
].copy()

print(f"Pairs before structure filtering: {len(pairs_mapped):,}")
print(f"Pairs after structure filtering:  {len(pairs_final):,}")
print(f"Pairs dropped:                    {len(pairs_mapped) - len(pairs_final):,}")

print(f"\nLabel distribution after filtering:")
print(pairs_final['label'].value_counts())
print(f"\nRatio: {(pairs_final['label']==0).sum() / (pairs_final['label']==1).sum():.4f}")
print(f"\nSource distribution:")
print(pairs_final['source'].value_counts())

Pairs before structure filtering: 35,686
Pairs after structure filtering:  7,324
Pairs dropped:                    28,362

Label distribution after filtering:
label
0    5625
1    1699
Name: count, dtype: int64

Ratio: 3.3108

Source distribution:
source
random_negative                      4705
STRING_experimental                  1699
Negatome_combined_stringent_human     920
Name: count, dtype: int64


In [ ]:
af2_failed_proteins = [
    uid for uid, v in structure_data.items()
    if v['source'] == 'AF2' and v['status'] == 'failed' and v['quality_note'] == 'download_failed'
]
print(f"Proteins to attempt with ESMFold: {len(af2_failed_proteins):,}")

Proteins to attempt with ESMFold: 1,656


In [ ]:
import requests
import time
import json

sequences = {}
seq_fetch_errors = []

for i, uid in enumerate(af2_failed_proteins):
    try:
        resp = requests.get(f"https://rest.uniprot.org/uniprotkb/{uid}.json", timeout=10)
        if resp.status_code == 200:
            seq = resp.json().get('sequence', {}).get('value', None)
            if seq:
                sequences[uid] = seq
            else:
                seq_fetch_errors.append(uid)
        else:
            seq_fetch_errors.append(uid)
    except Exception:
        seq_fetch_errors.append(uid)

    if (i+1) % 200 == 0:
        print(f"  Fetched {i+1}/{len(af2_failed_proteins)}...")
    time.sleep(0.1)

print(f"\nSequences retrieved: {len(sequences):,}")
print(f"Fetch errors: {len(seq_fetch_errors):,}")

# Save immediately
with open('/content/esmfold_sequences.json', 'w') as f:
    json.dump(sequences, f)

  Fetched 200/1656...
  Fetched 400/1656...
  Fetched 600/1656...
  Fetched 800/1656...
  Fetched 1000/1656...
  Fetched 1200/1656...
  Fetched 1400/1656...
  Fetched 1600/1656...

Sequences retrieved: 1,482
Fetch errors: 174


In [ ]:
seq_lengths = {uid: len(seq) for uid, seq in sequences.items()}

import numpy as np
lengths_array = np.array(list(seq_lengths.values()))

print(f"Sequence length statistics:")
print(f"  Min:    {lengths_array.min()}")
print(f"  Max:    {lengths_array.max()}")
print(f"  Mean:   {lengths_array.mean():.0f}")
print(f"  Median: {np.median(lengths_array):.0f}")
print(f"\nDistribution:")
print(f"  <= 300 residues:  {(lengths_array <= 300).sum()}")
print(f"  301-500:          {((lengths_array > 300) & (lengths_array <= 500)).sum()}")
print(f"  501-800:          {((lengths_array > 500) & (lengths_array <= 800)).sum()}")
print(f"  801-1500:         {((lengths_array > 800) & (lengths_array <= 1500)).sum()}")
print(f"  > 1500:           {(lengths_array > 1500).sum()}")

Sequence length statistics:
  Min:    51
  Max:    4911
  Mean:   630
  Median: 478

Distribution:
  <= 300 residues:  381
  301-500:          393
  501-800:          351
  801-1500:         265
  > 1500:           92


In [ ]:
MAX_LENGTH = 800

sequences_to_fold = {
    uid: seq for uid, seq in sequences.items() if len(seq) <= MAX_LENGTH
}

print(f"Proteins to fold with ESMFold: {len(sequences_to_fold):,}")
print(f"Proteins excluded (too long):  {len(sequences) - len(sequences_to_fold):,}")

Proteins to fold with ESMFold: 1,125
Proteins excluded (too long):  357


In [ ]:
import json

with open('/content/esmfold_sequences.json') as f:
    data = json.load(f)

print(f"Type: {type(data)}")
print(f"Number of entries: {len(data)}")
print(f"First few keys: {list(data.keys())[:5]}")

# Check if it's just IDs or actual sequences
first_key = list(data.keys())[0]
print(f"\nSample entry: {first_key} -> {data[first_key]}")

Type: <class 'dict'>
Number of entries: 1482
First few keys: ['Q5I022', 'O14690', 'Q53EZ3', 'O75047', 'Q8TCZ3']

Sample entry: Q5I022 -> MASKGAGMSFSRKSYRLTSDAEKSRVTGIVQEKLLNDYLNRIFSSSEHAPPAATSRKPLNFQNLPEHLDQLLQVDNEEEESQGQVEGRLGPSTVVLDHTGGFEGLLLVDDDLLGVIGHSNFGTIRSTTCVYKGKWLYEVLISSQGLMQIGWCTISCRFNQEEGVGDTHNSYAYDGNRVRKWNVTTTNYGKAWAAGDIVSCLIDLDDGTLSFCLNGVSLGTAFENLSRGLGMAYFPAISLSFKESVAFNFGSRPLRYPVAGYRPLQDPPSADLVRAQRLLGCFRAVLSVELDPVEGRLLDKESSKWRLRGQPTVLLTLAHIFHHFAPLLRKVYLVEAVLMSFLLGIVEKGTPTQAQSVVHQVLDLLWLFMEDYEVQDCLKQLMMSLLRLYRFSPIVPDLGLQIHYLRLTIAILRHEKSRKFLLSNVLFDVLRSVVFFYIKSPLRVEEAGLQELIPTTWWPHCSSREGKESTEMKEETAEERLRRRAYERGCQRLRKRIEVVEELQVQILKLLLDNKDDNGGEASRYIFLTKFRKFLQENASGRGNMPMLCPPEYMVCFLHRLISALRYYWDEYKASNPHASFSEEAYIPPQVFYNGKVDYFDLQRLGGLLSHLRKTLKDDLASKANIVIDPLELQSTAMDDLDEDEEPAPAMAQRPMQALAVGGPLPLPRPGWLSSPTLGRANRFLSTAAVSLMTPRRPLSTSEKVKVRTLSVEQRTREDIEGSHWNEGLLLGRPPEEPEQPLTENSLLEVLDGAVMMYNLSVHQQLGKMVGVSDDVNEYAMALRDTEDKLRRCPKRRKDILAELTKSQKVFSEKLDHLSRRLAWVHATVYSQEKMLDIYWLLRVCLRTIEHGDRTGSLFAFMP